In [ ]:
!pip install crewai -q
!pip install langchain -q
!py -m pip install -qU langchain-ollama
!pip install -U ollama
!py -m pip install markdown weasyprint -q
!py -m pip install -qU langchain
!py -m pip install --upgrade langchain langchain-core langchain-community
!py -m pip install langchain-classic

In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error,mean_absolute_percentage_error
import numpy as np
from xgboost import XGBRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from langchain.tools import tool
import shap
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from pathlib import Path

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
import json
import ast

In [22]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1:8b" 

llm = ChatOllama(
    model=MODEL,
    temperature=0
)

### Code base

In [ ]:
def preprocess(df, output_dir="D:/mycode/Fintech-agent/data/processed_data"):
    os.makedirs(output_dir, exist_ok=True)

    #make agent choose the target variable
    X = df.drop(columns=["caldt", "cpiret"])
    y = df["cpiret"]

    X_train       = X.iloc[:-8]
    X_test        = X.iloc[-8:]
    y_train       = y.iloc[:-8]
    y_test        = y.iloc[-8:]

    #may insert more preprocessing steps
    #is validation set needed ?
    X_train_path = os.path.join(output_dir, "X_train.csv")
    X_test_path = os.path.join(output_dir, "X_test.csv")

    y_train_path = os.path.join(output_dir, "y_train.csv")
    y_test_path = os.path.join(output_dir, "y_test.csv")

    X_train.to_csv(X_train_path, index=False)
    X_test.to_csv(X_test_path, index=False)

    y_train.to_csv(y_train_path, index=False)
    y_test.to_csv(y_test_path, index=False)

    paths = {
        "X_train": X_train_path,
        "X_test": X_test_path,
        "y_train": y_train_path,
        "y_test": y_test_path
    }

    return paths

In [ ]:
MODEL_REGISTRY = {
    # "xgb":   lambda cfg=None: XGBRegressor(
    #              n_estimators   = (cfg or {}).get("n_estimators",   800),
    #              learning_rate  = (cfg or {}).get("learning_rate",  0.05),
    #              max_depth      = (cfg or {}).get("max_depth",       6),
    #              subsample      = (cfg or {}).get("subsample",       0.8),
    #              colsample_bytree=(cfg or {}).get("colsample_bytree",0.8),
    #              reg_lambda     = (cfg or {}).get("reg_lambda",      1.0),
    #              objective      = "reg:squarederror",
    #              n_jobs=-1, random_state=42),
    "gbr":   lambda cfg=None: GradientBoostingRegressor(
                 n_estimators   = (cfg or {}).get("n_estimators",  300),
                 learning_rate  = (cfg or {}).get("learning_rate", 0.05),
                 max_depth      = (cfg or {}).get("max_depth",      3),
                 random_state=42),
    "svr":   lambda cfg=None: SVR(
                 kernel  = (cfg or {}).get("kernel",  "rbf"),
                 C       = (cfg or {}).get("C",        10.0),
                 gamma   = (cfg or {}).get("gamma",   "scale"),
                 epsilon = (cfg or {}).get("epsilon",  0.1)),
    "bayes": lambda cfg=None: BayesianRidge(),
    "knn":   lambda cfg=None: KNeighborsRegressor(
                 n_neighbors = (cfg or {}).get("n_neighbors", 5),
                 weights     = (cfg or {}).get("weights",     "distance"),
                 n_jobs=-1),
}

In [ ]:
def forecast(data_paths, model_name, model_config=None):
    X_train = pd.read_csv(data_paths["X_train"])
    X_test = pd.read_csv(data_paths["X_test"])

    y_train = pd.read_csv(data_paths["y_train"]).squeeze()
    y_test = pd.read_csv(data_paths["y_test"]).squeeze()

    model_name = str(model_name).lower().strip()
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model_name: {model_name}. Choose from {list(MODEL_REGISTRY.keys())}")

    model = MODEL_REGISTRY[model_name](model_config)
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    #calc SHAP
    if hasattr(model, "predict_proba") or "tree" in model_name or "forest" in model_name:
        explainer = shap.Explainer(model, X_train)
    else:
        explainer = shap.Explainer(model.predict, X_train)

    shap_values = explainer(X_test)

    feature_importance = (
        pd.DataFrame({
            "feature": X_test.columns,
            "importance": abs(shap_values.values).mean(axis=0)
        })
        .sort_values(by="importance", ascending=False)
        .reset_index(drop=True)
    )

    shap_result = feature_importance.to_dict(orient="records")

    return {
        "model_name": model_name,
        "model_config":  model_config or {},
        "train_r2": float(r2_score(y_train, y_train_pred)),
        "test_r2": float(r2_score(y_test, y_test_pred)),
        "test_rmse": float(np.sqrt(mean_squared_error(y_test, y_test_pred))),
        "test_mape": float(mean_absolute_percentage_error(y_test, y_test_pred) * 100),
        "shap_result": shap_result
    }

### Tools declaration

In [ ]:
@tool
def preprocess_tool(df_path: str) -> dict:
    """
    Preprocess dataset: split dataset, save splits as CSV files and return their file paths.
    """
    df = pd.read_csv(df_path)
    return preprocess(df)

In [27]:

@tool
def forecast_tool(data_paths: dict, model_name: str) -> dict:
    """
    Train a selected regression model using CSV paths and return evaluation metrics.
    model_name must be one of: gbr, svr, bayes, knn
    """
    return forecast(data_paths, model_name=model_name)

### agent orchestration

In [28]:
preprocess_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a data preprocessing agent.

You MUST call preprocess_tool with the provided dataset CSV path.

The tool will:
- split the dataset
- save processed datasets as CSV files

Return ONLY the dictionary of generated CSV file paths.
"""
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])


preprocess_agent = create_tool_calling_agent(
    llm,
    [preprocess_tool],
    preprocess_prompt
)


preprocess_executor = AgentExecutor(
    agent=preprocess_agent,
    tools=[preprocess_tool],
    verbose=True
)

In [ ]:

def build_forecast_prompt(history: list[dict] = []) -> ChatPromptTemplate:
    history_text = ""
    if history:
        history_text = "\n\nPREVIOUS ITERATION RESULTS:\n"
        for i, h in enumerate(history):
            history_text += f"""
                                Iteration {i+1}:
                                Model:      {h['model_name']}
                                Config:     {json.dumps(h.get('model_config', {}))}
                                Train R²:   {h['train_r2']}
                                Val R²:     {h['val_r2']}
                                Test R²:    {h['test_r2']}
                                Test RMSE:  {h['test_rmse']}
                                Top SHAP feature: {h['shap']['top_feature']}
                                SHAP ranking: {json.dumps(h['shap']['mean_abs_shap'])}
                                """
            history_text += """
                                OPTIMIZATION INSTRUCTIONS:
                                Based on the history above:
                                1. If train R² >> val R²: model is overfitting — increase regularization,
                                reduce max_depth, increase min_samples or reg_lambda.
                                2. If both train and val R² are low: model is underfitting — increase
                                n_estimators, reduce regularization, try a different model.
                                3. If val R² > test R²: data distribution shift — try a simpler model.
                                4. Pick the best model_name and suggest improved model_config hyperparameters.
                                5. Use SHAP ranking to consider if feature engineering could help.
                                
                                Suggest a model_config dict with updated hyperparameters to improve test R².
                                """
 
    system_msg = f"""
                    You are a forecasting agent.
                    You MUST call forecast_tool with:
                    - data_paths: dict with keys X_train, X_val, X_test, y_train, y_val, y_test
                    - model_name: one of xgb, gbr, svr, bayes, knn
                    - model_config: dict of hyperparameters (can be empty {{}} for defaults)
                    
                    {history_text}
                    Always call the tool. Return the metrics it produces.
                    """
 
    return ChatPromptTemplate.from_messages([
        ("system", system_msg),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}")
    ])

### Optimization Loop

In [ ]:

def run_optimization_loop(csv_path: str, max_iter: int):
    preprocess_result  = preprocess_executor.invoke({"input": csv_path})
    preprocessed_paths = preprocess_result["output"]
 
    history     = []
    best_metrics= None
    best_test_r2= -np.inf
 
    for iteration in range(1, max_iter + 1):
        print(f"\n{'='*65}")
        print(f"OPTIMIZATION ITERATION {iteration} / {max_iter}")
 
        forecast_prompt = build_forecast_prompt(history)
        forecast_agent  = create_tool_calling_agent(
            llm, [forecast_tool], forecast_prompt)
        forecast_executor = AgentExecutor(
            agent=forecast_agent,
            tools=[forecast_tool],
            verbose=True,
            handle_parsing_errors=True,
            return_intermediate_steps=True
        )
 
        forecast_result = forecast_executor.invoke(
            {"input": preprocessed_paths}
        )
 
        metrics = forecast_result["intermediate_steps"][-1][1]
 
        if not metrics:
            print(f"  WARNING: Could not parse metrics at iteration {iteration}, skipping.")
            continue
 
        # Sort SHAP by value
        if "shap_importance" in metrics:
            metrics["shap_importance"] = dict(
                sorted(metrics["shap_importance"].items(), 
                       key=lambda x: x[1], reverse=True))
    
 
        print(f"\n  Results:")
        print(f"  Model:     {metrics.get('model_name')}")
        print(f"  Config:    {metrics.get('model_config', {})}")
        print(f"  Train R²:  {metrics.get('train_r2')}")
        print(f"  Test R²:   {metrics.get('test_r2')}")
        print(f"  Test RMSE: {metrics.get('test_rmse')}")
 
        history.append(metrics)
 
        test_r2 = metrics.get("test_r2", -np.inf)
        if test_r2 > best_test_r2:
            best_test_r2  = test_r2
            best_metrics  = metrics
            print(f"  New best test R²: {best_test_r2:.4f}")
        else:
            print(f"  No improvement. Best so far: {best_test_r2:.4f}")
 
        # Early stop if R² is excellent
        if best_test_r2 >= 0.95:
            print(f"\n  Stopping early — test R² {best_test_r2:.4f} >= 0.95")
            break
 
    return best_metrics, history         

In [ ]:
best_metrics, history = run_optimization_loop(
        csv_path = r"D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv",
        max_iter = 2
    )

### Manual agent tests

In [ ]:
forecast_prompt = build_forecast_prompt()

forecast_agent = create_tool_calling_agent(
    llm,
    [forecast_tool],
    forecast_prompt
)


forecast_executor = AgentExecutor(
    agent=forecast_agent,
    tools=[forecast_tool],
    verbose=True,
    handle_parsing_errors=True,
    return_intermediate_steps=True
)

In [ ]:
# Step 1 — Preprocess dataset
preprocess_result = preprocess_executor.invoke(
    {"input": "D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv"}
)

preprocessed_paths = preprocess_result["output"]
print(preprocessed_paths)

In [ ]:
# Step 2 — Train model
forecast_result = forecast_executor.invoke(
    {"input": preprocessed_paths}
)

metrics = forecast_result["intermediate_steps"][-1][1]
print(metrics)

### Report generation Agent

In [ ]:
report_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a machine learning report generator.

Given evaluation metrics, generate a professional Markdown report including:

- Model Overview
- Performance Table
- Interpretation of metrics
- Overfitting Assessment
- Conclusion
"""
    ),
    ("human", "{input}")
])

report_chain = report_prompt | llm

In [ ]:
if "shap_importance" in metrics:
    metrics["shap_importance"] = dict(
        sorted(metrics["shap_importance"].items(), key=lambda x: x[1], reverse=True))
    

report_input = {
    "history":      history,
    "best_metrics": best_metrics,
    "iterations":   len(history)
}
# Step 3 — Generate report
report = report_chain.invoke(
    {"input": metrics} #best_metrics/report_input/metrics
)

print(report)

In [ ]:
with open("result/forecast_report.md", "w", encoding="utf-8") as f:
    f.write(report)

print("Report saved to forecast_report.md")


Report saved to forecast_report.md


### References
https://docs.langchain.com/oss/python/integrations/chat/ollama

https://docs.langchain.com/oss/python/langchain/tools


- feed more data about the forecasting process and feature engineering information for better report
- purpose of the task (is it to understand how agentic system works or should we try to build an efficient system for real life scenario)
- should the tool be called by the agent 
